In [ ]:
from cedne import simulator
from cedne import utils
import numpy as np

## Simulating a triad motif (FFI) with preset parameters and 1 step input

In [ ]:
triads = utils.return_triads()
G = triads["030T"]

weights = {(1, 3): -3.0, (3, 2): -1, (1, 2): -3}
input_nodes = [1]
gains = {node: 1.0 for node in G.nodes}
baselines = {node: 0.0 for node in G.nodes}
tconstants = [2, 2, 1]
time_constants = {n: t for n, t in zip(G.nodes, tconstants)}
initial_rates = [0.0, 0.0, 0.0]
time_points = np.linspace(0, 30, 90)

## First input
inp1_start = 30
inp1_end = 60
inp1_value = 1

input1 = simulator.StepInput(
    input_nodes,
    tstart=time_points[inp1_start],
    tend=time_points[inp1_end],
    value=inp1_value,
)
inputs = [input1]

rate_model = simulator.RateModel(
    G,
    input_nodes,
    weights,
    gains,
    time_constants,
    baselines,
    time_points=time_points,
    inputs=inputs,
)
rates = rate_model.simulate()
xticks = ([0, 10, 20, 30], ["0", "10", "20", "30"])
figsize = utils.mm_to_inches(
    (utils.plotting.single_column / 2.5, utils.plotting.single_column / 3)
)
fig = utils.plot_simulation_results(
    (rate_model, inputs, rates), figsize=figsize, xticks=xticks
)
fig.savefig("triad_simulation.svg", transparent=True)

In [ ]:
rate_model.neuron_parameters, rate_model.inputs

## Simulating a triad motif (FFI) with preset parameters: 2 adaptive inputs.

In [ ]:
## First input
inp1_start = 150
inp1_end = 300
inp1_value = 1
decay_rate_1 = 20  ## Adaptation rate of the receptor

## Second input
inp2_start = 300
inp2_end = 450
inp2_value = 0.2
decay_rate_2 = 20  ## Adaptation rate of the receptor

input1 = simulator.StepInputWithAdaptation(
    input_nodes,
    tstart=time_points[inp1_start],
    tend=time_points[inp1_end],
    value=inp1_value,
    decay_rate=decay_rate_1,
)
input2 = simulator.StepInputWithAdaptation(
    input_nodes,
    tstart=time_points[inp2_start],
    tend=time_points[inp2_end],
    value=inp2_value,
    decay_rate=decay_rate_2,
)
inputs = [input1, input2]

rate_model.remove_inputs()  # Removing previous inputs and adding adaptive inputs
rate_model.set_inputs(inputs)

rates = rate_model.simulate()
f = utils.plot_simulation_results((rate_model, inputs, rates))

In [ ]:
rate_model.neuron_parameters, rate_model.inputs

## Sinusiodal input with the input node of the graph with timeseries input data.

In [ ]:
triads = utils.return_triads()
G = triads["030T"]
weights = {(1, 3): -3.0, (3, 2): -1, (1, 2): -3}

input_nodes = [1]
gains = {node: 1.0 for node in G.nodes}
baselines = {node: 0.0 for node in G.nodes}
tconstants = [10, 10, 1]
time_constants = {n: t for n, t in zip(G.nodes, tconstants)}

initial_rates = [0.0, 0.0, 0.0]
max_t = 90
time_points = np.linspace(0, max_t, 451)

inp1_value = 1
input_value = {t: inp1_value * np.sin((t / max_t) * 2 * np.pi) for t in time_points}
inp_vals = [input_value[t] for t in time_points]
input1 = simulator.TimeSeriesInput(input_nodes, input_value)
inputs = [input1]

rate_model = simulator.RateModel(
    G,
    input_nodes,
    weights,
    gains,
    time_constants,
    baselines,
    static_neurons=input_nodes,
    inputs=inputs,
)
rates = rate_model.simulate(time_points=time_points)

f = utils.plot_simulation_results((rate_model, inputs, rates), twinx=False)

In [ ]:
rate_model.neuron_parameters, rate_model.edge_parameters, rate_model.inputs